In [ ]:
# ANALYSIS OF SHAP VALUES (PLOTTING)

# LIBRARIES

import numpy as np

import numpy as np
import xarray as xr

import matplotlib as mpl
from matplotlib import pyplot as plt
import colormaps as cmaps
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# TEST NN

# Path to network (.pt)
path_test = '/p/scratch/icon-a-ml/haslauer1/_data/_ERA5/_ML_new/_n32/SGIG16/_preprocessed_full2024_land_orostd_/__models/UNet_2_2026-04-15_UNet_2_SG_orog.pt'

# Number of levels
LEVS = 37

# Assign names to features
FEATURES_2D           = ['geop', 'oro_std', 'oro_anis', 'oro_angle', 'oro_slope']#, 'land_sea_mask']
feature_names = []
for i in range(LEVS):
    feature_names.append('u ' + str(i))
for i in range(LEVS):
    feature_names.append('v ' + str(i))
for i in range(LEVS):
    feature_names.append('w ' + str(i))
for i in range(LEVS):
    feature_names.append('t ' + str(i))
# for i in range(LEVS):
#     feature_names.append('hus ' + str(i))
# for i in range(LEVS):
#     feature_names.append('vort ' + str(i))
feature_names += FEATURES_2D[:]

In [ ]:
# SHAPLEY ALL LEVELS

shap_vals = np.load(path_test[:-3] + '_shap_vals.npy')
shap_vals.shape#, X_explain.shape

In [ ]:
# Compute absolute values

shap_mean = np.mean(np.abs(shap_vals), axis=0)
shap_mean = shap_mean/np.sum(shap_mean, axis=0)
shap_mean.shape

In [ ]:
# SHAP values per classes of features

shap_classes = np.stack([np.sum(shap_mean[0:37,:], axis=0)*100,
                         np.sum(shap_mean[37:74,:], axis=0)*100,
                         np.sum(shap_mean[74:111,:], axis=0)*100,
                         np.sum(shap_mean[111:148,:], axis=0)*100,
                         np.sum(shap_mean[148:,:], axis=0)*100])
                        #  np.sum(shap_mean[148:185,:], axis=0),
                        #  np.sum(shap_mean[185:222,:], axis=0),
                        #  np.sum(shap_mean[222:223,:], axis=0),
                        #  np.sum(shap_mean[223:228,:], axis=0),
                        #  np.sum(shap_mean[228:229,:], axis=0)])

shap_classes.shape

In [ ]:
# PLOT SHAP VALUES ON LEVELS

era5_levels = [1.0, 2.0, 3.0, 5.0, 7.0, 
        10.0, 20.0, 30.0, 50.0, 70.0, 
        100.0, 125.0, 150.0, 175.0, 200.0, 225.0, 250.0,
        300.0, 350.0, 400.0, 450.0, 500.0, 550.0, 600.0, 650.0,
        700.0, 750.0, 775.0, 800.0, 825.0, 850.0, 875.0, 900.0, 925.0, 950.0, 975.0, 1000.0]

color = mpl.cm.get_cmap(cmaps.naviaw)

In [ ]:
# PLOT FOR ZONAL FLUXES

fig = plt.figure(figsize=(30,12), dpi=150)

plt.rcParams.update({'font.size': 29})

fs1 = 32
fs2 = 29
labelpad = 7
pad=24

width, height = 0.1, 0.84
#pad=18.5

ax00 = fig.add_axes([0.055,  0.092,  0.23, height])
ax01 = fig.add_axes([0.343,  0.092,  0.23, height])
ax02 = fig.add_axes([0.750,  0.092,  0.23, height])

formatter = FuncFormatter(lambda y, _: f'{int(y)}')

linewidth = 2.7

ax00.plot(shap_mean[:37,5], era5_levels, label='$u$', color = cmaps.paired(1), linewidth=linewidth)
ax00.plot(shap_mean[37:74,5], era5_levels, label='$v$', color = cmaps.paired(3), linewidth=linewidth)
ax00.plot(shap_mean[74:111,5], era5_levels, label='$\omega$', color = cmaps.paired(7), linewidth=linewidth)
ax00.plot(shap_mean[111:148,5], era5_levels, label='$T$', color = cmaps.paired(5), linewidth=linewidth)

ax01.plot(shap_mean[:37,9], era5_levels, label='$u$', color = cmaps.paired(1), linewidth=linewidth)
ax01.plot(shap_mean[37:74,9], era5_levels, label='$v$', color = cmaps.paired(3), linewidth=linewidth)
ax01.plot(shap_mean[74:111,9], era5_levels, label='$\omega$', color = cmaps.paired(7), linewidth=linewidth)
ax01.plot(shap_mean[111:148,9], era5_levels, label='$T$', color = cmaps.paired(5), linewidth=linewidth)

ax02.plot(shap_mean[148,   :37], era5_levels, label='$z$', color = cmaps.paired(1), linewidth=linewidth)
ax02.plot(shap_mean[149, :37], era5_levels, label='$\mu$', color = cmaps.paired(3), linewidth=linewidth)
ax02.plot(shap_mean[150, :37], era5_levels, label='$\gamma$', color = cmaps.paired(7), linewidth=linewidth)
ax02.plot(shap_mean[151,:37], era5_levels, label='$\sigma$', color = cmaps.paired(5), linewidth=linewidth)
ax02.plot(shap_mean[152,:37], era5_levels, label=r'$\theta$', color = cmaps.paired(8), linewidth=linewidth)

ax00.invert_yaxis()
ax00.grid()
ax00.set_yscale('log')
ax00.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax00.set_ylabel('Input level pressure [hPa]',fontsize=fs2, labelpad=0)
ax00.set_xlim([0,0.15])
ax00.set_ylim([1000,1])
ax00.axhline(10, color='tab:grey',linewidth=4)
ax00.set_xticks([0.03,0.06,0.09,0.12,0.15])
ax00.yaxis.set_major_formatter(formatter)
ax00.set_title('(a) 10 hPa', fontsize=fs1, pad=pad)
ax00.tick_params(axis='both', which='major', labelsize=fs2)
ax00.legend(fontsize=fs2)

ax01.invert_yaxis()
ax01.grid()
ax01.set_yscale('log')
ax01.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax01.set_xlim([0,0.15])
ax01.set_ylim([1000,1])
ax01.axhline(70, color='tab:grey',linewidth=4)
ax01.set_xticks([0.00,0.03,0.06,0.09,0.12,0.15])
ax01.yaxis.set_major_formatter(formatter)
ax01.set_title('(b) 70 hPa', fontsize=fs1, pad=pad)
ax01.tick_params(axis='both', which='major', labelsize=fs2, left=False, labelleft=False)
ax01.legend(fontsize=fs2)

ax02.invert_yaxis()
ax02.grid()
ax02.set_yscale('log')
ax02.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax02.set_ylabel('Output level pressure [hPa]',fontsize=fs2, labelpad=0)
ax02.set_xlim([0,0.15])
ax02.set_ylim([1000,1])
ax02.set_xticks([0.03,0.06,0.09,0.12,0.15])
ax02.yaxis.set_major_formatter(formatter)
ax02.set_title('(c) Orographic variables', fontsize=fs1, pad=pad)
ax02.tick_params(axis='both', which='major', labelsize=fs2)
ax02.legend(fontsize=fs2-2)

#plt.savefig('/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260101_SHAP_levels_IG_zonal.png', dpi=150)

In [ ]:
# PLOT FOR MERIDIONAL FLUXES

fig = plt.figure(figsize=(30,12), dpi=150)

plt.rcParams.update({'font.size': 29})

fs1 = 32
fs2 = 29
labelpad = 7
pad=24

width, height = 0.1, 0.84
#pad=18.5

ax00 = fig.add_axes([0.055,  0.092,  0.23, height])
ax01 = fig.add_axes([0.343,  0.092,  0.23, height])
ax02 = fig.add_axes([0.750,  0.092,  0.23, height])

formatter = FuncFormatter(lambda y, _: f'{int(y)}')

linewidth = 2.7

ax00.plot(shap_mean[:37,42], era5_levels, label='$u$', color = cmaps.paired(1), linewidth=linewidth)
ax00.plot(shap_mean[37:74,42], era5_levels, label='$v$', color = cmaps.paired(3), linewidth=linewidth)
ax00.plot(shap_mean[74:111,42], era5_levels, label='$\omega$', color = cmaps.paired(7), linewidth=linewidth)
ax00.plot(shap_mean[111:148,42], era5_levels, label='$T$', color = cmaps.paired(5), linewidth=linewidth)

ax01.plot(shap_mean[:37,46], era5_levels, label='$u$', color = cmaps.paired(1), linewidth=linewidth)
ax01.plot(shap_mean[37:74,46], era5_levels, label='$v$', color = cmaps.paired(3), linewidth=linewidth)
ax01.plot(shap_mean[74:111,46], era5_levels, label='$\omega$', color = cmaps.paired(7), linewidth=linewidth)
ax01.plot(shap_mean[111:148,46], era5_levels, label='$T$', color = cmaps.paired(5), linewidth=linewidth)

ax02.plot(shap_mean[148,   37:], era5_levels, label='$z$', color = cmaps.paired(1), linewidth=linewidth)
ax02.plot(shap_mean[149, 37:], era5_levels, label='$\mu$', color = cmaps.paired(3), linewidth=linewidth)
ax02.plot(shap_mean[150, 37:], era5_levels, label='$\gamma$', color = cmaps.paired(7), linewidth=linewidth)
ax02.plot(shap_mean[151, 37:], era5_levels, label='$\sigma$', color = cmaps.paired(5), linewidth=linewidth)
ax02.plot(shap_mean[152, 37:], era5_levels, label=r'$\theta$', color = cmaps.paired(8), linewidth=linewidth)

ax00.invert_yaxis()
ax00.grid()
ax00.set_yscale('log')
ax00.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax00.set_ylabel('Input level pressure [hPa]',fontsize=fs2, labelpad=0)
ax00.set_xlim([0,0.15])
ax00.set_ylim([1000,1])
ax00.axhline(10, color='tab:grey',linewidth=4)
ax00.set_xticks([0.03,0.06,0.09,0.12,0.15])
ax00.yaxis.set_major_formatter(formatter)
ax00.set_title('(a) 10 hPa', fontsize=fs1, pad=pad)
ax00.tick_params(axis='both', which='major', labelsize=fs2)
ax00.legend(fontsize=fs2)

ax01.invert_yaxis()
ax01.grid()
ax01.set_yscale('log')
ax01.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax01.set_xlim([0,0.15])
ax01.set_ylim([1000,1])
ax01.axhline(70, color='tab:grey',linewidth=4)
ax01.set_xticks([0.00,0.03,0.06,0.09,0.12,0.15])
ax01.yaxis.set_major_formatter(formatter)
ax01.set_title('(b) 70 hPa', fontsize=fs1, pad=pad)
ax01.tick_params(axis='both', which='major', labelsize=fs2, left=False, labelleft=False)
ax01.legend(fontsize=fs2)

ax02.invert_yaxis()
ax02.grid()
ax02.set_yscale('log')
ax02.set_xlabel('SHAP importance',fontsize=fs2, labelpad=labelpad)
ax02.set_ylabel('Output level pressure [hPa]',fontsize=fs2, labelpad=0)
ax02.set_xlim([0,0.15])
ax02.set_ylim([1000,1])
ax02.set_xticks([0.03,0.06,0.09,0.12,0.15])
ax02.yaxis.set_major_formatter(formatter)
ax02.set_title('(c) Orographic variables', fontsize=fs1, pad=pad)
ax02.tick_params(axis='both', which='major', labelsize=fs2)
ax02.legend(fontsize=fs2-2)

#plt.savefig('/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260101_SHAP_levels_IG_merid.png', dpi=150)

In [ ]:
# Calculate mean SHAP values

shaps = np.mean(np.abs(shap_vals), axis=-1)
X_explain = np.load(path_test[:-3] + '_X_explain.npy')

mean_sv_3d = shap_mean[:148,:].reshape(4, LEVS, 2, LEVS)
mean_sv_all = shap_mean
mean_sv_all_new = mean_sv_all/(mean_sv_all.max())

mean_sv_3d.shape, mean_sv_all.shape, mean_sv_all.mean(axis=1).shape, mean_sv_all_new.shape

In [ ]:
# Assemble matrix to plot

plotmatrix = np.zeros_like(mean_sv_all_new)

plotmatrix[  :37,  :37] = mean_sv_all_new[  :37,  :37][::-1,:]
plotmatrix[  :37,37:74] = mean_sv_all_new[  :37,37:74][::-1,:]


plotmatrix[37:74,  :37] = mean_sv_all_new[37:74,  :37][::-1,:]
plotmatrix[37:74,37:74] = mean_sv_all_new[37:74,37:74][::-1,:]

plotmatrix[74:111,  :37] = mean_sv_all_new[74:111,  :37][::-1,:]
plotmatrix[74:111,37:74] = mean_sv_all_new[74:111,37:74][::-1,:]


plotmatrix[111:148,  :37] = mean_sv_all_new[111:148,  :37][::-1,:]
plotmatrix[111:148,37:74] = mean_sv_all_new[111:148,37:74][::-1,:]

plotmatrix[148:,:] = mean_sv_all_new[148:,:]

In [ ]:
# SHAP PLOT FOR CLASSES AND MATRIX

plt.rcParams.update({'font.size': 29})
fs2 = 29

x_ext, y_ext = mean_sv_all_new.shape[0], mean_sv_all_new.shape[1]

fig = plt.figure(figsize=(30,9.5), dpi=150)

ax0 = fig.add_axes([0.04,  0.06,  0.2, 0.9])
ax1 = fig.add_axes([0.29,  0.06,  0.665, 0.9])

cm1 = cmaps.paired(0.3)

ax0.bar(['u','v','$\omega$','T','orog'], np.mean(shap_classes, axis=1), color=cm1)
ax0.set_ylabel('Relative importance (%)', fontsize=fs2, labelpad=10)
ax0.set_ylim([0,27.5])

abs_max = np.max(np.abs(mean_sv_3d))
print('abs_max: ', abs_max)
abs_max = 0.1

cmap = cmaps.naviaw_r

p = [-abs_max, -0.1*abs_max, 0.1*abs_max, abs_max]
f = lambda x: np.interp(x, p, [0, 0.5, 0.5, 1])


def transform_x(x):
    x = np.asarray(x)
    return np.where(x <= 147.5, x, 147.5 + (x - 147.5) * 2.25)

X_orig_edges = np.arange(-0.5, 153.5, 1)  # Boundaries from -0.5 to 152.5
X_edges = transform_x(X_orig_edges)
Y_edges = np.arange(-0.5, 74.5, 1)        # Boundaries from -0.5 to 73.5

mean_shap_values3d_plot_2d = plotmatrix.reshape(x_ext, y_ext)

im = ax1.pcolormesh(X_edges, Y_edges, mean_shap_values3d_plot_2d.transpose(),
                    cmap=cmap, 
                    shading='flat')
ax1.invert_yaxis()

divider = make_axes_locatable(ax1)
cax1 = divider.append_axes("right", size=0.35, pad=0.35)
cbar = fig.colorbar(im, cax=cax1, orientation='vertical')
cbar.set_label(label='Mean absolute SHAP value', fontsize=fs2, labelpad=12)

plot_diff = 0.5

for idx in range(1,2,1):
    ax1.axhline(37*idx-plot_diff, ls='--', color='tab:blue', lw=1)
for idx in range(1,5,1):
    orig_x = 37*idx-plot_diff
    ax1.axvline(transform_x(orig_x), ls='--', color='tab:blue', lw=1)

for idx in range(74):
    ax1.axhline(idx-plot_diff, ls='--', color='tab:blue', lw=0.1)
for idx in range(153):
    orig_x = idx-plot_diff
    ax1.axvline(transform_x(orig_x), ls='--', color='tab:blue', lw=0.1)

#X_tick_labels = ['$u$', '$v$', '$\omega$', '$T$', 'orog']

X_tick_labels = ['$u$', '$v$', '$\omega$', '$T$', r'$\mathcal{z}$', r'$\mathcal{\mu}$', r'$\mathcal{\gamma}$', r'$\mathcal{\sigma}$', r'$\mathcal{\theta}$']
Y_tick_labels = ['$MF_x\,$ ', '$MF_y\,$ ']

tick_offset = 18.5

#X_ticks_orig = np.array([0+tick_offset, 37+tick_offset, 74+tick_offset, 111+tick_offset, 148+2])

X_ticks_orig = np.array([0+tick_offset, 37+tick_offset, 74+tick_offset, 111+tick_offset, 148, 149, 150, 151, 152])

X_ticks = transform_x(X_ticks_orig)

ax1.set_xticks(X_ticks, X_tick_labels)
Y_ticks = [0+tick_offset, 37+tick_offset]
ax1.set_yticks(Y_ticks, Y_tick_labels)

plt.savefig('/p/project1/icon-a-ml/haslauer1/____project1/_publication_4500/_20260608_SHAP_SG.png', dpi=150)
plt.show()